## Project Overview

This project implements a Retrieval-Augmented Generation (RAG) system for medical knowledge.

The system uses a collection of medical PDF documents covering common diseases. The documents are processed, divided into smaller text chunks, converted into vector embeddings, and stored in a ChromaDB vector database.

When a user asks a question, the system retrieves the most relevant chunks from the medical knowledge base and provides them as context to a language model. The model then generates an answer based only on the retrieved context and includes the source document used.

### Domain

Medical Knowledge / Common Diseases

### Dataset

The dataset consists of 4 medical PDF documents:

- Autoimmune_disease.pdf
- Hypertension.pdf
- Diabetes.pdf
- Cardiovascular_disease.pdf

The dataset contains a total of 146 pages.

### Main Technologies

- Python
- Jupyter Notebook
- pypdf
- Sentence Transformers
- ChromaDB
- Ollama
- Llama 3.2 3B

### RAG Pipeline

The main pipeline consists of:

1. Loading and extracting text from the medical PDFs.
2. Checking the extracted text and determining whether OCR is required.
3. Splitting the documents into overlapping text chunks.
4. Generating embeddings using `all-MiniLM-L6-v2`.
5. Storing the chunks and embeddings in ChromaDB.
6. Retrieving relevant chunks for a user question.
7. Providing the retrieved context to the language model.
8. Generating an answer with source citations.
9. Evaluating the system using 10 test questions.
10. Exporting the persistent ChromaDB vector store for backend integration.

# 2. Load & Inspect

## Load PDFs

In [1]:
from pathlib import Path
from pypdf import PdfReader

DATA_DIR = Path("../data")

pdf_files = list(DATA_DIR.glob("*.pdf"))

print(f"Number of PDF files: {len(pdf_files)}")

for pdf in pdf_files:
    print(pdf.name)

Number of PDF files: 4
Autoimmune_disease.pdf
Cardiovascular_disease.pdf
Diabetes.pdf
Hypertension.pdf


## Count documents

In [3]:
for pdf in pdf_files:
    reader = PdfReader(pdf)
    print(f"{pdf.name}: {len(reader.pages)} pages")

Autoimmune_disease.pdf: 22 pages
Cardiovascular_disease.pdf: 44 pages
Diabetes.pdf: 40 pages
Hypertension.pdf: 40 pages


## Count pages

## Extract text

In [4]:
documents_text = {}

for pdf in pdf_files:
    reader = PdfReader(pdf)

    text = ""
    for page in reader.pages:
        page_text = page.extract_text() or ""
        text += page_text + "\n"

    documents_text[pdf.name] = text

    print(f"{pdf.name}: {len(text):,} characters")

Autoimmune_disease.pdf: 70,620 characters
Cardiovascular_disease.pdf: 161,371 characters
Diabetes.pdf: 134,801 characters
Hypertension.pdf: 131,280 characters


## Check parsing/OCR

In [5]:
for filename, text in documents_text.items():
    print("=" * 80)
    print(filename)
    print("=" * 80)
    print(text[:1000])
    print("\n")

Autoimmune_disease.pdf
Autoimmune diseases
Young woman with malar rash, typically found
in systemic lupus erythematosus
Specialty Rheumatology, immunology,
gastroenterology, neurology,
dermatology, endocrinology
Symptoms Wide-ranging, depends on the
condition. Commonly include,
although by no means restricted
to, low grade fever, feeling tired[1]
Usual
onset
Adulthood[1]
Types List of autoimmune diseases
(alopecia areata, vitiligo, celiac
disease, diabetes mellitus type 1,
Hashimoto's disease, Graves'
disease, inflammatory bowel
disease, multiple sclerosis,
psoriasis, rheumatoid arthritis,
systemic lupus erythematosus,
others)[1]
Medication Nonsteroidal anti-inflammatory
drugs, immunosuppressants,
intravenous immunoglobulin[1][2]
Frequency 10% (UK)[3]
Autoimmune disease
An autoimmune disease is a condition causing disease
that results from an anomalous response of the
adaptive immune system, wherein it mistakenly targets
and attacks healthy, functioning parts of the body as if
they wer

### Parsing / OCR Check

All 4 PDF files were successfully parsed using `pypdf`.

The extracted text is readable and contains meaningful content from the source documents. Therefore, OCR is not required for the current dataset.

Total documents: 4  
Total pages: 146

The extracted text will be used for the next step: chunking.

3. Chunking Strategy

The extracted text is divided into smaller chunks before generating embeddings.

Initial configuration:
- Chunk size: 800 characters
- Overlap: 150 characters

### Rationale

A chunk size of 800 characters provides enough surrounding context for medical concepts while keeping retrieved passages relatively focused.

An overlap of 150 characters helps preserve context across chunk boundaries so that information split between two consecutive chunks is less likely to lose its meaning.

These values are used as an initial configuration and can be adjusted based on retrieval performance during evaluation.

## Split text

In [16]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150


def create_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        # Move the end slightly backward to avoid cutting a word
        if end < len(text):
            while end > start and not text[end].isspace():
                end -= 1

        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        # Move backward for overlap
        next_start = max(0, end - overlap)

        # Move forward to the beginning of the next word
        while next_start < end and not text[next_start].isspace():
            next_start += 1

        while next_start < len(text) and text[next_start].isspace():
            next_start += 1

        if next_start <= start:
            next_start = end

        start = next_start

    return chunks

## Chunk size



The final chunk size was set to 800 characters.

This size provides enough context for medical information while keeping each retrieved passage focused and manageable.

In [17]:
all_chunks = []

for filename, text in documents_text.items():
    chunks = create_chunks(text)

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "source": filename,
            "chunk_id": i,
            "text": chunk
        })

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 770


In [18]:
for i, chunk in enumerate(all_chunks[:3]):
    print("=" * 80)
    print(f"Source: {chunk['source']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print(f"Characters: {len(chunk['text'])}")
    print("=" * 80)
    print(chunk["text"])
    print()

Source: Autoimmune_disease.pdf
Chunk ID: 0
Characters: 794
Autoimmune diseases
Young woman with malar rash, typically found
in systemic lupus erythematosus
Specialty Rheumatology, immunology,
gastroenterology, neurology,
dermatology, endocrinology
Symptoms Wide-ranging, depends on the
condition. Commonly include,
although by no means restricted
to, low grade fever, feeling tired[1]
Usual
onset
Adulthood[1]
Types List of autoimmune diseases
(alopecia areata, vitiligo, celiac
disease, diabetes mellitus type 1,
Hashimoto's disease, Graves'
disease, inflammatory bowel
disease, multiple sclerosis,
psoriasis, rheumatoid arthritis,
systemic lupus erythematosus,
others)[1]
Medication Nonsteroidal anti-inflammatory
drugs, immunosuppressants,
intravenous immunoglobulin[1][2]
Frequency 10% (UK)[3]
Autoimmune disease
An autoimmune disease is a condition

Source: Autoimmune_disease.pdf
Chunk ID: 1
Characters: 800
drugs, immunosuppressants,
intravenous immunoglobulin[1][2]
Frequency 10% (UK)[3]
Auto

## Overlap justification



A 150-character overlap is used between consecutive chunks.

The overlap helps preserve context when a sentence or related piece of information occurs near a chunk boundary. The implementation also avoids starting the overlap in the middle of a word.

Final configuration:
- Chunk size: 800 characters
- Overlap: 150 characters
- Total chunks: 770

4. Embeddings

## Embedding Model

We use the `all-MiniLM-L6-v2` sentence-transformer model to convert each text chunk into a numerical vector.

The model is lightweight and suitable for semantic similarity and retrieval tasks. Using the same embedding model for both document chunks and user queries ensures that they are represented in the same vector space.

## Load embedding model

In [19]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\LAPTOP\Downloads\rag-medical-assistant\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LAPTOP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


## Generate embeddings

In [20]:
texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True
)

print("Embeddings generated successfully.")
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimension: {embeddings.shape[1]}")

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Embeddings generated successfully.
Number of embeddings: 770
Embedding dimension: 384


5. ## Vector Store

ChromaDB is used as the vector database for storing the text chunks and their embeddings.

Each chunk is stored with its source filename and chunk ID as metadata. The vector store is persisted locally so it can be loaded later by the backend application.

## Chroma

In [21]:
import chromadb
from pathlib import Path

CHROMA_PATH = "../chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(
    name="medical_rag"
)

print("Chroma collection created successfully.")
print(f"Collection name: {collection.name}")

Chroma collection created successfully.
Collection name: medical_rag


## Store embeddings

In [22]:
ids = [
    f"{chunk['source']}_{chunk['chunk_id']}"
    for chunk in all_chunks
]

metadatas = [
    {
        "source": chunk["source"],
        "chunk_id": chunk["chunk_id"]
    }
    for chunk in all_chunks
]

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Documents and embeddings stored successfully.")
print(f"Number of items in collection: {collection.count()}")

Documents and embeddings stored successfully.
Number of items in collection: 770


## Persist to Disk

The ChromaDB vector store is persisted locally in the `chroma_db` directory.

The collection is named `medical_rag` and contains 770 document chunks with their embeddings and source metadata.

This persisted vector store will be used later by the backend application.

In [23]:
print(f"ChromaDB path: {CHROMA_PATH}")
print(f"Collection name: {collection.name}")
print(f"Stored items: {collection.count()}")

ChromaDB path: ../chroma_db
Collection name: medical_rag
Stored items: 770


6. ## Retrieval

The retrieval step searches the ChromaDB vector store for the most semantically relevant chunks to a given question.

The question is embedded using the same embedding model used for the document chunks. ChromaDB then returns the most relevant documents and their metadata.

## Retrieval function

In [24]:
def retrieve_documents(question, top_k=5):
    query_embedding = embedding_model.encode([question])[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results

## Test Retrieval

A test question was used to verify that the retrieval system returns relevant chunks from the medical corpus.

For the question about common symptoms of autoimmune diseases, the top retrieved results were from `Autoimmune_disease.pdf` and contained relevant symptom information.

This confirms that the vector store can retrieve semantically relevant medical content.

In [25]:
question = "What are the common symptoms of autoimmune diseases?"

results = retrieve_documents(question, top_k=5)

for i, document in enumerate(results["documents"][0]):
    print("=" * 80)
    print(f"Result {i + 1}")
    print(f"Source: {results['metadatas'][0][i]['source']}")
    print(f"Chunk ID: {results['metadatas'][0][i]['chunk_id']}")
    print(document[:500])
    print()

Result 1
Source: Autoimmune_disease.pdf
Chunk ID: 8
normal. It is profound and
prevents [them] from doing the simplest everyday tasks." and 59% said it was "probably the
most debilitating symptom of having an [autoimmune disease]."[13]
low-grade fever
Signs and symptoms
Common symptoms
malaise (a general feeling of discomfort or unease)
muscle aches
joint pain
skin rashes
Autoimmune diseases can present a diverse array of symptoms. For instance, some people may
experience dry mouth or dry eyes, tingling or numbness in various body parts, unexpect

Result 2
Source: Autoimmune_disease.pdf
Chunk ID: 10
and unexplained weight loss.
Commonly affected areas in autoimmune diseases include blood vessels, connective tissues, joints,
muscles, red blood cells, skin, and endocrine glands such as the thyroid gland (in diseases like
Hashimoto's thyroiditis and Graves' disease) and the pancreas (in type 1 diabetes). The impacts of these
diseases can range from localized damage to certain tissues, alt

## 7. RAG Prompt

The RAG prompt combines the user's question with the relevant retrieved context.

The language model is instructed to answer using only the provided context and to cite the source documents used for the answer.

If the provided context does not contain enough information to answer the question, the model should clearly state that the information is not available in the retrieved context.

## Context

In [26]:
def build_context(results):
    context_parts = []

    for i, document in enumerate(results["documents"][0]):
        source = results["metadatas"][0][i]["source"]
        chunk_id = results["metadatas"][0][i]["chunk_id"]

        context_parts.append(
            f"[Source: {source}, Chunk: {chunk_id}]\n{document}"
        )

    return "\n\n".join(context_parts)

## Question

In [27]:
def build_rag_prompt(question, context):
    prompt = f"""
You are a medical knowledge assistant.

Answer the question using only the information provided in the context below.

If the context does not contain enough information to answer the question, say that the information is not available in the provided context.

Always include the source document name in your answer.

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

## Citations

In [28]:
question = "What are the common symptoms of autoimmune diseases?"

results = retrieve_documents(question, top_k=5)

context = build_context(results)

prompt = build_rag_prompt(question, context)

print(prompt)


You are a medical knowledge assistant.

Answer the question using only the information provided in the context below.

If the context does not contain enough information to answer the question, say that the information is not available in the provided context.

Always include the source document name in your answer.

Context:
[Source: Autoimmune_disease.pdf, Chunk: 8]
normal. It is profound and
prevents [them] from doing the simplest everyday tasks." and 59% said it was "probably the
most debilitating symptom of having an [autoimmune disease]."[13]
low-grade fever
Signs and symptoms
Common symptoms
malaise (a general feeling of discomfort or unease)
muscle aches
joint pain
skin rashes
Autoimmune diseases can present a diverse array of symptoms. For instance, some people may
experience dry mouth or dry eyes, tingling or numbness in various body parts, unexpected changes in
weight, and diarrhea.
These symptoms often reflect the body's systemic inflammatory response. However, their occur

In [29]:
import ollama

def generate_answer(prompt):
    response = ollama.chat(
        model="llama3.2:3b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response["message"]["content"]

In [30]:
answer = generate_answer(prompt)

print(answer)

According to the provided context, the common symptoms of autoimmune diseases include:

* Malaise (a general feeling of discomfort or unease)
* Muscle aches
* Joint pain
* Skin rashes
* Dry mouth or dry eyes
* Tingling or numbness in various body parts
* Unexpected changes in weight
* Diarrhea
* Fatigue (reported by 98% of people with autoimmune diseases)

These symptoms can vary in intensity and occurrence over time, and can be influenced by factors such as age, sex, hormonal status, and environmental influences.

(Source: Autoimmune_disease.pdf, Chunk: 6 and 10)


8.evaluation


We evaluate the RAG system using 10 questions covering different topics from the medical documents.

The questions are designed to test whether the system retrieves relevant information and generates answers supported by the provided context.

## 10 Questions

In [31]:
evaluation_questions = [
    "What are the common symptoms of autoimmune diseases?",
    "What are the main risk factors for hypertension?",
    "What are common symptoms of diabetes?",
    "What are the major risk factors for cardiovascular disease?",
    "What are some complications of diabetes?",
    "How can hypertension affect the body?",
    "What are some types of autoimmune diseases?",
    "What are common symptoms of cardiovascular disease?",
    "What are some ways to manage diabetes?",
    "What are some ways to prevent cardiovascular disease?"
]

print(f"Number of evaluation questions: {len(evaluation_questions)}")

Number of evaluation questions: 10


## Retrieved Sources

In [32]:
evaluation_results = []

for question in evaluation_questions:
    # Retrieve relevant documents
    results = retrieve_documents(question, top_k=5)

    # Build context
    context = build_context(results)

    # Build RAG prompt
    rag_prompt = build_rag_prompt(question, context)

    # Generate answer
    answer = generate_answer(rag_prompt)

    # Collect retrieved sources
    sources = []
    for metadata in results["metadatas"][0]:
        source = metadata["source"]
        chunk_id = metadata["chunk_id"]
        sources.append(f"{source} (Chunk {chunk_id})")

    evaluation_results.append({
        "question": question,
        "answer": answer,
        "sources": sources
    })

print(f"Evaluation completed for {len(evaluation_results)} questions.")

Evaluation completed for 10 questions.


## Retrieved sources


In [33]:
for i, result in enumerate(evaluation_results, start=1):
    print("=" * 80)
    print(f"Question {i}: {result['question']}")
    print("Retrieved sources:")

    for source in result["sources"]:
        print(f"- {source}")

    print()

Question 1: What are the common symptoms of autoimmune diseases?
Retrieved sources:
- Autoimmune_disease.pdf (Chunk 8)
- Autoimmune_disease.pdf (Chunk 10)
- Autoimmune_disease.pdf (Chunk 6)
- Autoimmune_disease.pdf (Chunk 0)
- Autoimmune_disease.pdf (Chunk 7)

Question 2: What are the main risk factors for hypertension?
Retrieved sources:
- Hypertension.pdf (Chunk 0)
- Hypertension.pdf (Chunk 1)
- Hypertension.pdf (Chunk 2)
- Hypertension.pdf (Chunk 17)
- Hypertension.pdf (Chunk 98)

Question 3: What are common symptoms of diabetes?
Retrieved sources:
- Diabetes.pdf (Chunk 1)
- Diabetes.pdf (Chunk 5)
- Diabetes.pdf (Chunk 0)
- Diabetes.pdf (Chunk 12)
- Diabetes.pdf (Chunk 8)

Question 4: What are the major risk factors for cardiovascular disease?
Retrieved sources:
- Cardiovascular_disease.pdf (Chunk 9)
- Cardiovascular_disease.pdf (Chunk 10)
- Cardiovascular_disease.pdf (Chunk 2)
- Cardiovascular_disease.pdf (Chunk 35)
- Cardiovascular_disease.pdf (Chunk 77)

Question 5: What are some

## Answers

In [34]:
for i, result in enumerate(evaluation_results, start=1):
    print("=" * 80)
    print(f"Question {i}")
    print(f"Question: {result['question']}")
    print("\nAnswer:")
    print(result["answer"])
    print()

Question 1
Question: What are the common symptoms of autoimmune diseases?

Answer:
According to the provided context from the Autoimmune_disease.pdf document, the common symptoms of autoimmune diseases include:

* Fatigue (most common complaint, 98% of people with autoimmune diseases experience fatigue)
* Malaise (a general feeling of discomfort or unease)
* Muscle aches
* Joint pain
* Skin rashes
* Dry mouth or dry eyes
* Tingling or numbness in various body parts
* Unexpected changes in weight
* Diarrhea
* Low-grade fever
* Unexplained weight loss

These symptoms can vary in intensity and frequency based on the type of disease, organ systems affected, and individual factors such as age, sex, hormonal status, and environmental influences.

Question 2
Question: What are the main risk factors for hypertension?

Answer:
The main risk factors for hypertension, as stated in the provided context, include:

1. Lack of sleep
2. Excess salt in the diet
3. Excess body weight
4. Smoking
5. Alcoh

## Correct / Incorrect

Each generated answer was manually evaluated by comparing it with the retrieved context and the information available in the source documents.

An answer is marked as Correct when it is supported by the retrieved context and addresses the question appropriately. Otherwise, it is marked as Incorrect.

In [35]:
import pandas as pd

evaluation_table = []

for i, result in enumerate(evaluation_results, start=1):
    evaluation_table.append({
        "Question": result["question"],
        "Answer": result["answer"],
        "Retrieved Sources": "; ".join(result["sources"]),
        "Correct / Incorrect": ""
    })

evaluation_df = pd.DataFrame(evaluation_table)

evaluation_df

,Question,Answer,Retrieved Sources,Correct / Incorrect
0,What are the common symptoms of autoimmune dis...,According to the provided context from the Aut...,Autoimmune_disease.pdf (Chunk 8); Autoimmune_d...,
1,What are the main risk factors for hypertension?,"The main risk factors for hypertension, as sta...",Hypertension.pdf (Chunk 0); Hypertension.pdf (...,
2,What are common symptoms of diabetes?,Common symptoms of diabetes include increased ...,Diabetes.pdf (Chunk 1); Diabetes.pdf (Chunk 5)...,
3,What are the major risk factors for cardiovasc...,"According to the provided context, particularl...",Cardiovascular_disease.pdf (Chunk 9); Cardiova...,
4,What are some complications of diabetes?,"According to the provided context, specificall...",Diabetes.pdf (Chunk 1); Diabetes.pdf (Chunk 8)...,
5,How can hypertension affect the body?,"According to the provided context, hypertensio...",Hypertension.pdf (Chunk 14); Hypertension.pdf ...,
6,What are some types of autoimmune diseases?,Some types of autoimmune diseases include:\n\n...,Autoimmune_disease.pdf (Chunk 0); Autoimmune_d...,
7,What are common symptoms of cardiovascular dis...,"According to the provided context, the common ...",Cardiovascular_disease.pdf (Chunk 0); Cardiova...,
8,What are some ways to manage diabetes?,"According to the provided source document, ""Di...",Diabetes.pdf (Chunk 41); Diabetes.pdf (Chunk 3...,
9,What are some ways to prevent cardiovascular d...,"Based on the provided context, some ways to pr...",Cardiovascular_disease.pdf (Chunk 51); Cardiov...,


In [36]:
for i, result in enumerate(evaluation_results, start=1):
    print("=" * 80)
    print(f"Question {i}: {result['question']}")
    print("\nAnswer:")
    print(result["answer"])
    print()

Question 1: What are the common symptoms of autoimmune diseases?

Answer:
According to the provided context from the Autoimmune_disease.pdf document, the common symptoms of autoimmune diseases include:

* Fatigue (most common complaint, 98% of people with autoimmune diseases experience fatigue)
* Malaise (a general feeling of discomfort or unease)
* Muscle aches
* Joint pain
* Skin rashes
* Dry mouth or dry eyes
* Tingling or numbness in various body parts
* Unexpected changes in weight
* Diarrhea
* Low-grade fever
* Unexplained weight loss

These symptoms can vary in intensity and frequency based on the type of disease, organ systems affected, and individual factors such as age, sex, hormonal status, and environmental influences.

Question 2: What are the main risk factors for hypertension?

Answer:
The main risk factors for hypertension, as stated in the provided context, include:

1. Lack of sleep
2. Excess salt in the diet
3. Excess body weight
4. Smoking
5. Alcohol use

These risk

In [37]:
for i, result in enumerate(evaluation_results, start=1):
    print(f"\n{'=' * 60}")
    print(f"QUESTION {i}")
    print(f"{'=' * 60}")
    print(result["question"])
    print("\nANSWER:")
    print(result["answer"])
    print("\nRETRIEVED SOURCES:")
    for source in result["sources"]:
        print(f"- {source}")


QUESTION 1
What are the common symptoms of autoimmune diseases?

ANSWER:
According to the provided context from the Autoimmune_disease.pdf document, the common symptoms of autoimmune diseases include:

* Fatigue (most common complaint, 98% of people with autoimmune diseases experience fatigue)
* Malaise (a general feeling of discomfort or unease)
* Muscle aches
* Joint pain
* Skin rashes
* Dry mouth or dry eyes
* Tingling or numbness in various body parts
* Unexpected changes in weight
* Diarrhea
* Low-grade fever
* Unexplained weight loss

These symptoms can vary in intensity and frequency based on the type of disease, organ systems affected, and individual factors such as age, sex, hormonal status, and environmental influences.

RETRIEVED SOURCES:
- Autoimmune_disease.pdf (Chunk 8)
- Autoimmune_disease.pdf (Chunk 10)
- Autoimmune_disease.pdf (Chunk 6)
- Autoimmune_disease.pdf (Chunk 0)
- Autoimmune_disease.pdf (Chunk 7)

QUESTION 2
What are the main risk factors for hypertension?

A

In [38]:
print("QUESTION 2:")
print(evaluation_results[1]["question"])

print("\nANSWER:")
print(evaluation_results[1]["answer"])

print("\nSOURCES:")
for source in evaluation_results[1]["sources"]:
    print("-", source)

QUESTION 2:
What are the main risk factors for hypertension?

ANSWER:
The main risk factors for hypertension, as stated in the provided context, include:

1. Lack of sleep
2. Excess salt in the diet
3. Excess body weight
4. Smoking
5. Alcohol use

These risk factors are mentioned in the "Risk factors" section of the Hypertension.pdf document, specifically in Chunk: 0.

SOURCES:
- Hypertension.pdf (Chunk 0)
- Hypertension.pdf (Chunk 1)
- Hypertension.pdf (Chunk 2)
- Hypertension.pdf (Chunk 17)
- Hypertension.pdf (Chunk 98)


In [39]:
print("QUESTION 3:")
print(evaluation_results[2]["question"])

print("\nANSWER:")
print(evaluation_results[2]["answer"])

print("\nSOURCES:")
for source in evaluation_results[2]["sources"]:
    print("-", source)

QUESTION 3:
What are common symptoms of diabetes?

ANSWER:
Common symptoms of diabetes include increased thirst, frequent urination, extreme hunger, and unexplained weight loss. (Source: Diabetes.pdf, Chunk: 5)

SOURCES:
- Diabetes.pdf (Chunk 1)
- Diabetes.pdf (Chunk 5)
- Diabetes.pdf (Chunk 0)
- Diabetes.pdf (Chunk 12)
- Diabetes.pdf (Chunk 8)


In [40]:
print("QUESTION 4:")
print(evaluation_results[3]["question"])

print("\nANSWER:")
print(evaluation_results[3]["answer"])

print("\nSOURCES:")
for source in evaluation_results[3]["sources"]:
    print("-", source)

QUESTION 4:
What are the major risk factors for cardiovascular disease?

ANSWER:
According to the provided context, particularly in [Source: Cardiovascular_disease.pdf, Chunk: 2], the major risk factors for cardiovascular disease include:

* High blood pressure
* Smoking
* Diabetes mellitus
* Lack of exercise
* Obesity
* High blood cholesterol
* Poor diet
* Excessive alcohol consumption
* Poor sleep
* Family history/genetic predisposition
* Age
* Sex

These risk factors are modifiable by lifestyle change, social change, and drug treatment. (Source: Cardiovascular_disease.pdf, Chunk: 24)

Additionally, the context also mentions that existing cardiovascular disease or a previous cardiovascular event is the strongest predictor of a future cardiovascular event. (Source: Cardiovascular_disease.pdf, Chunk: 77)

Note that the context also mentions that genetics is an important risk factor for cardiovascular diseases, but it is not explicitly listed as a separate risk factor. (Source: Cardiova

In [41]:
print("QUESTION 5:")
print(evaluation_results[4]["question"])

print("\nANSWER:")
print(evaluation_results[4]["answer"])

print("\nSOURCES:")
for source in evaluation_results[4]["sources"]:
    print("-", source)

QUESTION 5:
What are some complications of diabetes?

ANSWER:
According to the provided context, specifically [Source: Diabetes.pdf, Chunk: 30-33], some complications of diabetes include:

- Damage to blood vessels at both macrovascular and microvascular levels
- Cardiovascular disease, which doubles the risk of heart-related issues
- Stroke
- Peripheral artery disease
- Microvascular disease affecting the eyes, kidneys, and nerves

These complications can lead to health issues such as disorders of the cardiovascular system, eye, kidney, and nerves, and can even be fatal in severe cases.

SOURCES:
- Diabetes.pdf (Chunk 1)
- Diabetes.pdf (Chunk 8)
- Diabetes.pdf (Chunk 26)
- Diabetes.pdf (Chunk 61)
- Diabetes.pdf (Chunk 3)


In [42]:
print("QUESTION 6:")
print(evaluation_results[5]["question"])

print("\nANSWER:")
print(evaluation_results[5]["answer"])

print("\nSOURCES:")
for source in evaluation_results[5]["sources"]:
    print("-", source)

QUESTION 6:
How can hypertension affect the body?

ANSWER:
According to the provided context, hypertension can affect the body in various ways. The symptoms of hypertension are not explicitly stated in the provided context, but it mentions that hypertension is a major risk factor for several conditions, including:

* Coronary artery disease
* Stroke
* Heart failure
* Peripheral arterial disease
* Vision loss
* Chronic kidney disease
* Dementia

Additionally, the text states that hypertension can cause certain specific signs and symptoms, such as:

* Truncal obesity (Cushing's syndrome)
* Glucose intolerance
* Moon face
* Buffalo hump
* Purple abdominal stretch marks (Cushing's syndrome)
* Weight loss with increased appetite (Hyperthyroidism)
* Fast heart rate
* Bulging eyes
* Tremor (Hyperthyroidism)
* Decreased blood pressure in the lower extremities relative to the arms (Coarctation of the aorta)
* Delayed or absent femoral arterial pulses (Coarctation of the aorta)

It is also menti

In [43]:
print("QUESTION 7:")
print(evaluation_results[6]["question"])

print("\nANSWER:")
print(evaluation_results[6]["answer"])

print("\nSOURCES:")
for source in evaluation_results[6]["sources"]:
    print("-", source)

QUESTION 7:
What are some types of autoimmune diseases?

ANSWER:
Some types of autoimmune diseases include:

1. Alopecia areata
2. Vitiligo
3. Celiac disease
4. Diabetes mellitus type 1
5. Hashimoto's disease
6. Graves' disease
7. Inflammatory bowel disease
8. Multiple sclerosis
9. Psoriasis
10. Rheumatoid arthritis
11. Systemic lupus erythematosus
12. Others (as listed in the document)

Source: Autoimmune_disease.pdf, Chunk: 1

SOURCES:
- Autoimmune_disease.pdf (Chunk 0)
- Autoimmune_disease.pdf (Chunk 1)
- Autoimmune_disease.pdf (Chunk 3)
- Autoimmune_disease.pdf (Chunk 109)
- Autoimmune_disease.pdf (Chunk 6)


In [44]:
print("QUESTION 8:")
print(evaluation_results[7]["question"])

print("\nANSWER:")
print(evaluation_results[7]["answer"])

print("\nSOURCES:")
for source in evaluation_results[7]["sources"]:
    print("-", source)

QUESTION 8:
What are common symptoms of cardiovascular disease?

ANSWER:
According to the provided context, the common symptoms of cardiovascular disease are:

* Chest pain
* Shortness of breath
* Fatigue
* Loss of consciousness

(Source: Cardiovascular_disease.pdf, Chunk: 0)

SOURCES:
- Cardiovascular_disease.pdf (Chunk 0)
- Cardiovascular_disease.pdf (Chunk 6)
- Cardiovascular_disease.pdf (Chunk 5)
- Hypertension.pdf (Chunk 0)
- Hypertension.pdf (Chunk 58)


In [45]:
print("QUESTION 9:")
print(evaluation_results[8]["question"])

print("\nANSWER:")
print(evaluation_results[8]["answer"])

print("\nSOURCES:")
for source in evaluation_results[8]["sources"]:
    print("-", source)

QUESTION 9:
What are some ways to manage diabetes?

ANSWER:
According to the provided source document, "Diabetes.pdf", some ways to manage diabetes include:

1. Maintaining adequate glycemic control (Source: Diabetes.pdf, Chunk: 38)
2. Keeping blood sugar levels close to normal, without causing low blood sugar (Source: Diabetes.pdf, Chunk: 38)
3. Dietary changes, such as maintaining a diet rich in whole grains and fiber, and choosing good fats (Source: Diabetes.pdf, Chunk: 38)
4. Exercise, such as engaging in physical activity of more than 90 minutes per day, which reduces the risk of diabetes by 28% (Source: Diabetes.pdf, Chunk: 38)
5. Weight loss
6. Use of appropriate medications (insulin, oral medications) (Source: Diabetes.pdf, Chunk: 38)
7. Education and learning about the disease, actively participating in treatment, and managing other health problems that may accelerate the negative effects of diabetes (Source: Diabetes.pdf, Chunk: 42)
8. Glucose control, with the goal of mainta

In [46]:
print("QUESTION 10:")
print(evaluation_results[9]["question"])

print("\nANSWER:")
print(evaluation_results[9]["answer"])

print("\nSOURCES:")
for source in evaluation_results[9]["sources"]:
    print("-", source)

QUESTION 10:
What are some ways to prevent cardiovascular disease?

ANSWER:
Based on the provided context, some ways to prevent cardiovascular disease include:

1. Maintaining a healthy diet, such as the Mediterranean diet, a vegetarian, vegan, or another plant-based diet.
2. Avoiding established risk factors.

Additionally, there are several risk stratification models, but their sensitivity for population groups and impact analysis are unclear. Instead, future preventative screening appears to shift toward applying prevention according to randomized trial results of each intervention.

Source: Cardiovascular_disease.pdf, Chunk: 51 and 140.

SOURCES:
- Cardiovascular_disease.pdf (Chunk 51)
- Cardiovascular_disease.pdf (Chunk 140)
- Cardiovascular_disease.pdf (Chunk 185)
- Cardiovascular_disease.pdf (Chunk 224)
- Cardiovascular_disease.pdf (Chunk 90)


In [49]:
evaluation_labels = [
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct",
    "Correct"
]

print("Number of labels:", len(evaluation_labels))
print("Number of questions:", len(evaluation_df))

evaluation_df["Correct / Incorrect"] = evaluation_labels

evaluation_df

Number of labels: 10
Number of questions: 10


,Question,Answer,Retrieved Sources,Correct / Incorrect
0,What are the common symptoms of autoimmune dis...,According to the provided context from the Aut...,Autoimmune_disease.pdf (Chunk 8); Autoimmune_d...,Correct
1,What are the main risk factors for hypertension?,"The main risk factors for hypertension, as sta...",Hypertension.pdf (Chunk 0); Hypertension.pdf (...,Correct
2,What are common symptoms of diabetes?,Common symptoms of diabetes include increased ...,Diabetes.pdf (Chunk 1); Diabetes.pdf (Chunk 5)...,Correct
3,What are the major risk factors for cardiovasc...,"According to the provided context, particularl...",Cardiovascular_disease.pdf (Chunk 9); Cardiova...,Correct
4,What are some complications of diabetes?,"According to the provided context, specificall...",Diabetes.pdf (Chunk 1); Diabetes.pdf (Chunk 8)...,Correct
5,How can hypertension affect the body?,"According to the provided context, hypertensio...",Hypertension.pdf (Chunk 14); Hypertension.pdf ...,Correct
6,What are some types of autoimmune diseases?,Some types of autoimmune diseases include:\n\n...,Autoimmune_disease.pdf (Chunk 0); Autoimmune_d...,Correct
7,What are common symptoms of cardiovascular dis...,"According to the provided context, the common ...",Cardiovascular_disease.pdf (Chunk 0); Cardiova...,Correct
8,What are some ways to manage diabetes?,"According to the provided source document, ""Di...",Diabetes.pdf (Chunk 41); Diabetes.pdf (Chunk 3...,Correct
9,What are some ways to prevent cardiovascular d...,"Based on the provided context, some ways to pr...",Cardiovascular_disease.pdf (Chunk 51); Cardiov...,Correct


In [52]:
correct_count = (evaluation_df["Correct / Incorrect"] == "Correct").sum()
total_count = len(evaluation_df)
accuracy = correct_count / total_count

print(f"Correct answers: {correct_count}/{total_count}")
print(f"Accuracy: {accuracy:.2%}")

Correct answers: 10/10
Accuracy: 100.00%


# 9. Failure Analysis

### Failure Analysis

The evaluation results showed that all 10 test questions received correct answers based on the retrieved context.

No major factual failures were observed during the evaluation. However, some minor limitations were identified:

- In some cases, the retrieved results included additional chunks that were less directly relevant to the question.
- Some generated answers included extra information beyond what was necessary to answer the question.
- Increasing retrieval precision or adjusting the number of retrieved chunks could help make the answers more focused.

Overall, the RAG pipeline successfully retrieved relevant medical information and generated answers supported by the provided context.

# 10. Export

The ChromaDB vector store is stored locally in the `chroma_db` directory.

The collection `medical_rag` contains the embedded document chunks and their metadata. The vector store will be shared with the backend developer for integration with the RAG application.

## Vector Store

In [50]:
from pathlib import Path

chroma_path = Path("../chroma_db")

print("ChromaDB path:", chroma_path.resolve())
print("Path exists:", chroma_path.exists())

ChromaDB path: C:\Users\LAPTOP\Downloads\rag-medical-assistant\chroma_db
Path exists: True


In [51]:
import chromadb

test_client = chromadb.PersistentClient(
    path=str(chroma_path)
)

test_collection = test_client.get_collection(
    name="medical_rag"
)

print("Collection loaded successfully.")
print("Collection name:", test_collection.name)
print("Stored items:", test_collection.count())

Collection loaded successfully.
Collection name: medical_rag
Stored items: 770
